<a href="https://colab.research.google.com/github/bsyeol/26AI_project/blob/main/notebooks/model_E4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# [E4-0] 패키지 설치
# - B0과 같은 TabPFN 버전을 설치합니다. 버전이 다르면 공정한 비교가 아닙니다.
# - B0 노트북에서 확인한 버전과 다르면 숫자를 맞춰 바꾸세요.
# - 런타임 유형이 GPU인지 먼저 확인하세요.
# =====================================================================
!pip install -q tabpfn==9.2.0

In [2]:
# =====================================================================
# [E4-1] 공통 데이터 불러오기 (B0-1과 동일)
# - 데이터 정리 노트북이 저장한 파일을 읽고 MD5로 같은 파일인지 확인합니다.
# - settings.json 값(SEEDS, K, H_DIM, FLIP_R, T2V_K, EPOCHS 등)을 변수로 만듭니다.
# - 원본에 없는 feature 두 개(log_amount, dow)를 만듭니다.
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, time, copy
import numpy as np
import pandas as pd

MID_DIR = '/content/drive/MyDrive/26-2_AI_project/processed/mid'
info = json.load(open(f'{MID_DIR}/split_info.json', encoding='utf-8'))
S    = json.load(open(f'{MID_DIR}/settings.json', encoding='utf-8'))
globals().update(S)

csv_path = f'{MID_DIR}/mid_v1.csv'
assert hashlib.md5(open(csv_path, 'rb').read()).hexdigest() == info['md5'], '파일이 다릅니다'

mid = pd.read_csv(csv_path, encoding='utf-8-sig',
                  dtype={USER: str, '입금계좌일련번호': str}, parse_dates=['date'])
mid['log_amount'] = np.log1p(mid['거래금액'])
mid['dow'] = mid['date'].dt.dayofweek

CONTEXT = {s: np.load(f'{MID_DIR}/context_seed{s}.npy') for s in SEEDS}

print(mid['split'].value_counts().to_dict(), '| 기대값:', info['rows'])
print('context:', {s: len(v) for s, v in CONTEXT.items()})

Mounted at /content/drive
{'train': 229653, 'test': 50264, 'val': 49404} | 기대값: {'train': 229653, 'test': 50264, 'val': 49404}
context: {0: 9996, 1: 9996, 2: 9996}


In [3]:
# =====================================================================
# [E4-2] TabPFN, PyTorch 준비
# - 보안 비밀 TABPFN_TOKEN을 읽어 라이선스 인증을 합니다.
# - GPU를 확인합니다. 인코더 학습도 GPU에서 해야 시간이 맞습니다.
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             f1_score, recall_score, precision_score)

from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN').strip()
from tabpfn import TabPFNClassifier
import tabpfn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| tabpfn', tabpfn.__version__)

device: cuda | tabpfn 9.2.0


In [4]:
# =====================================================================
# [E4-3] 평가 함수와 TabPFN 실행 함수 (B0과 동일 + Precision 추가)
# - evaluate: PR-AUC를 val, test에서 계산합니다.
#   F1, Recall, Precision은 val에서 F1이 가장 높은 threshold를 찾아 test에 적용합니다.
# - run_tabpfn: seed별 고정 context로 TabPFN을 실행합니다.
#   B0과 다른 것은 feats(입력 컬럼 목록)뿐입니다.
# =====================================================================
def evaluate(p_val, y_val, p_test, y_test):
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    t = thr[np.argmax(f1[:-1])]
    pred = (p_test >= t).astype(int)
    return {
        'val_pr_auc':     average_precision_score(y_val, p_val),
        'test_pr_auc':    average_precision_score(y_test, p_test),
        'test_f1':        f1_score(y_test, pred),
        'test_recall':    recall_score(y_test, pred),
        'test_precision': precision_score(y_test, pred, zero_division=0),
        'threshold':      float(t),
    }

def run_tabpfn(feats, seed):
    ctx = CONTEXT[seed]
    clf = TabPFNClassifier(random_state=seed)
    clf.fit(mid.loc[ctx, feats].to_numpy(np.float32), mid.loc[ctx, LABEL].to_numpy())

    def proba(part):
        X = mid.loc[mid['split'] == part, feats].to_numpy(np.float32)
        return np.concatenate([clf.predict_proba(X[i:i + PRED_BATCH])[:, 1]
                               for i in range(0, len(X), PRED_BATCH)])

    y_val  = mid.loc[mid['split'] == 'val',  LABEL].to_numpy()
    y_test = mid.loc[mid['split'] == 'test', LABEL].to_numpy()
    return evaluate(proba('val'), y_val, proba('test'), y_test)

In [5]:
# =====================================================================
# [E4-4] (계좌, 3시간 bin) 집계
# - 같은 계좌, 같은 3시간 구간의 거래를 한 행으로 묶습니다.
#   같은 구간 안의 거래 순서는 알 수 없으므로 순서 없이 요약합니다.
# - cnt: 건수 / amt_sum, amt_mean, amt_max: 금액 합, 평균, 최대
#   n_recv: 고유 수취계좌 수 / n_inst: 고유 입금기관 수
#   gap: 같은 계좌의 직전 거래 구간 이후 지난 구간 수
#   bin_of_day, dow: 시간대(0~7), 요일(0~6) -> Time2Vec 입력
# - 라벨(이상거래여부)은 집계에 쓰지 않습니다.
# =====================================================================
bins = (mid.groupby([USER, 'bin_id'])
          .agg(cnt=('date', 'size'),
               amt_sum=('거래금액', 'sum'), amt_mean=('거래금액', 'mean'), amt_max=('거래금액', 'max'),
               n_recv=('입금계좌일련번호', 'nunique'), n_inst=('입금금융회사일련번호', 'nunique'),
               bin_of_day=('bin_of_day', 'first'), date=('date', 'first'), split=('split', 'first'))
          .reset_index()
          .sort_values([USER, 'bin_id'], kind='stable')
          .reset_index(drop=True))
bins['gap'] = bins.groupby(USER)['bin_id'].diff().fillna(0)
bins['dow'] = bins['date'].dt.dayofweek

print('bin 수:', len(bins), '| 계좌 수:', bins[USER].nunique())
print('계좌당 bin 수:', bins.groupby(USER).size().describe(percentiles=[.25, .5, .75]).round(1).to_dict())

bin 수: 287660 | 계좌 수: 27694
계좌당 bin 수: {'count': 27694.0, 'mean': 10.4, 'std': 10.6, 'min': 1.0, '25%': 3.0, '50%': 8.0, '75%': 15.0, 'max': 204.0}


In [6]:
# =====================================================================
# [E4-5] 정규화
# - 건수, 금액, 간격을 log 변환한 뒤 표준화합니다.
# - 평균과 표준편차는 train 구간에서만 계산합니다(val, test 정보 누수 방지).
# - X: 시퀀스 feature (bin 수 x 7), T: Time2Vec 입력 (bin 수 x 2)
# =====================================================================
SEQ_FEATS = ['cnt', 'amt_sum', 'amt_mean', 'amt_max', 'n_recv', 'n_inst', 'gap']

Z = np.log1p(bins[SEQ_FEATS].to_numpy(np.float64))
is_tr = (bins['split'] == 'train').to_numpy()
mu, sd = Z[is_tr].mean(0), Z[is_tr].std(0) + 1e-6
X = ((Z - mu) / sd).astype(np.float32)
T = bins[['bin_of_day', 'dow']].to_numpy(np.float32)
print('train 기준 평균:', dict(zip(SEQ_FEATS, np.round(mu, 2))))

train 기준 평균: {'cnt': np.float64(0.73), 'amt_sum': np.float64(11.16), 'amt_mean': np.float64(11.11), 'amt_max': np.float64(11.12), 'n_recv': np.float64(0.73), 'n_inst': np.float64(0.72), 'gap': np.float64(2.77)}


In [7]:
# =====================================================================
# [E4-6] 시퀀스 창 만들기
# - 각 (계좌, bin) 행마다 같은 계좌의 바로 이전 K개 bin의 행 번호를 모읍니다.
# - 현재 bin은 넣지 않습니다(strict). 판단 시점에 알 수 있는 정보만 씁니다.
# - 이력이 K개보다 짧으면 앞쪽을 0 벡터로 채웁니다(행 번호 -1).
# - 사전학습 정답은 현재 bin의 집계값입니다("이전 K개로 다음 bin 예측").
# - TR_ROWS: train 구간 학습 샘플, VA_ROWS: val 구간 early stopping용 샘플
# =====================================================================
N = len(bins)
pos = bins.groupby(USER).cumcount().to_numpy()
WIN = np.full((N, K), -1, dtype=np.int64)
for k in range(1, K + 1):
    ok = pos >= k
    WIN[ok, K - k] = np.arange(N)[ok] - k

X_pad = np.vstack([X, np.zeros((1, X.shape[1]), np.float32)])
T_pad = np.vstack([T, np.zeros((1, T.shape[1]), np.float32)])

def batch(rows):
    w = WIN[rows]
    return (torch.from_numpy(X_pad[w]).to(DEVICE),
            torch.from_numpy(T_pad[w]).to(DEVICE),
            torch.from_numpy(X[rows]).to(DEVICE))

has_hist = pos >= 1
TR_ROWS = np.where(has_hist & (bins['split'] == 'train').to_numpy())[0]
VA_ROWS = np.where(has_hist & (bins['split'] == 'val').to_numpy())[0]
print('사전학습 train:', len(TR_ROWS), '| val:', len(VA_ROWS))
print('이력 없는 bin 비율 (계좌의 첫 bin):', f'{(pos == 0).mean():.1%}')
print(f'이력이 K={K}개 이상인 bin 비율:', f'{(pos >= K).mean():.1%}')

사전학습 train: 174973 | val: 42251
이력 없는 bin 비율 (계좌의 첫 bin): 9.6%
이력이 K=16개 이상인 bin 비율: 19.3%


In [8]:
# =====================================================================
# [E4-7] 인코더 정의: Time2Vec + SIGMA
# - Time2Vec: 시간대, 요일 -> 선형 항 1개 + 학습 가능한 주기(sin) 항.
# - Mamba: selective SSM. 입력에 따라 상태 갱신 정도(dt)가 달라집니다. PyTorch만으로 구현.
# - SIGMA 블록 (Liu et al., AAAI 2025)
#   PF-Mamba: 원래 순서 Mamba + Partial Flip(최근 FLIP_R개 유지, 나머지 역순) Mamba
#   DS Gate: 두 Mamba 출력의 가중치를 입력에 따라 결정
#   FE-GRU: Conv1d + GRU로 짧은 구간 패턴
#   Mixing -> Linear -> PFFN -> 잔차 + LayerNorm
# - Time2Vec은 Partial Flip 이전에 더합니다. 순서를 뒤집어도 각 bin의 시각 정보는 유지됩니다.
# - 출력 h: 마지막 위치(가장 최근 bin)를 H_DIM 차원으로 바꾼 벡터
# - 파라미터 수가 PARAM_BUDGET 범위 밖이면 멈춥니다(모델 간 크기 통일).
# =====================================================================
class Time2Vec(nn.Module):
    def __init__(self, in_dim, k):
        super().__init__()
        self.lin = nn.Linear(in_dim, 1)
        self.per = nn.Linear(in_dim, k - 1)
    def forward(self, t):
        return torch.cat([self.lin(t), torch.sin(self.per(t))], -1)


class Mamba(nn.Module):
    def __init__(self, d, n=16, expand=2, k=4):
        super().__init__()
        di = d * expand
        self.di, self.n = di, n
        self.inp  = nn.Linear(d, 2 * di)
        self.conv = nn.Conv1d(di, di, k, groups=di, padding=k - 1)
        self.xp   = nn.Linear(di, 1 + 2 * n)
        self.dt   = nn.Linear(1, di)
        self.A    = nn.Parameter(-torch.rand(di, n))
        self.out  = nn.Linear(di, d)

    def forward(self, x):
        B, L, _ = x.shape
        u, z = self.inp(x).chunk(2, -1)
        u = F.silu(self.conv(u.transpose(1, 2))[..., :L].transpose(1, 2))
        p = self.xp(u)
        dt, Bm, Cm = p[..., :1], p[..., 1:1 + self.n], p[..., 1 + self.n:]
        dt = F.softplus(self.dt(dt))
        h = x.new_zeros(B, self.di, self.n)
        ys = []
        for t in range(L):
            h = torch.exp(dt[:, t, :, None] * self.A) * h + dt[:, t, :, None] * Bm[:, t, None, :] * u[:, t, :, None]
            ys.append((h * Cm[:, t, None, :]).sum(-1))
        return self.out(torch.stack(ys, 1) * F.silu(z))


class SigmaEncoder(nn.Module):
    def __init__(self, f_in, t_in, d=H_DIM, r=FLIP_R, t2v_k=T2V_K):
        super().__init__()
        self.r = r
        self.emb   = nn.Linear(f_in, d)
        self.t2v   = Time2Vec(t_in, t2v_k)
        self.t_emb = nn.Linear(t2v_k, d)
        self.m_fwd, self.m_flip = Mamba(d), Mamba(d)
        self.g_conv = nn.Conv1d(d, d, 3, padding=1)
        self.g_lin  = nn.Linear(d, d)
        self.f_conv = nn.Conv1d(d, d, 3, padding=1)
        self.gru    = nn.GRU(d, d, batch_first=True)
        self.a      = nn.Parameter(torch.ones(2))
        self.lin    = nn.Linear(d, d)
        self.ffn    = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.norm   = nn.LayerNorm(d)
        self.proj   = nn.Linear(d, H_DIM)

    def gate(self, h):
        g = self.g_lin(self.g_conv(h.transpose(1, 2)).transpose(1, 2))
        return F.silu(g) + torch.sigmoid(g)

    def forward(self, x, t):
        h = self.emb(x) + self.t_emb(self.t2v(t))
        n = h.shape[1] - self.r
        h_flip = torch.cat([h[:, :n].flip(1), h[:, n:]], 1)
        M = self.gate(h) * self.m_fwd(h) + self.gate(h_flip) * self.m_flip(h_flip)
        Fg = self.gru(self.f_conv(h.transpose(1, 2)).transpose(1, 2))[0]
        z = self.lin(self.a[0] * M + self.a[1] * Fg)
        out = self.norm(h + self.ffn(z))
        return self.proj(out[:, -1])


n_params = sum(p.numel() for p in SigmaEncoder(len(SEQ_FEATS), 2).parameters())
print('인코더 파라미터 수:', n_params)
assert PARAM_BUDGET[0] <= n_params <= PARAM_BUDGET[1], '파라미터 수가 허용 범위 밖입니다'

인코더 파라미터 수: 44540


In [9]:
# =====================================================================
# [E4-8] 인코더 사전학습과 h 추출 함수
# - train_encoder: 이전 K개 bin으로 현재 bin 집계값을 예측하도록 학습(Huber loss).
#   이상거래 라벨은 쓰지 않습니다. train 구간 bin으로만 학습합니다.
#   val loss가 PATIENCE번 연속 나아지지 않으면 멈추고 가장 좋았던 가중치로 되돌립니다.
#   학습 후 예측 head는 버리고 인코더만 반환합니다.
# - extract_h: 모든 (계좌, bin)에 대해 h를 계산합니다.
# =====================================================================
def train_encoder(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    enc  = SigmaEncoder(len(SEQ_FEATS), 2).to(DEVICE)
    head = nn.Linear(H_DIM, len(SEQ_FEATS)).to(DEVICE)
    opt  = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=LR)

    def val_loss():
        enc.eval(); head.eval(); tot = 0.0
        with torch.no_grad():
            for i in range(0, len(VA_ROWS), BATCH):
                xs, ts, y = batch(VA_ROWS[i:i + BATCH])
                tot += F.huber_loss(head(enc(xs, ts)), y, reduction='sum').item()
        return tot / (len(VA_ROWS) * len(SEQ_FEATS))

    best, best_state, wait = float('inf'), None, 0
    for ep in range(EPOCHS):
        enc.train(); head.train(); t0 = time.time()
        perm = rng.permutation(TR_ROWS)
        for i in range(0, len(perm), BATCH):
            xs, ts, y = batch(perm[i:i + BATCH])
            loss = F.huber_loss(head(enc(xs, ts)), y)
            opt.zero_grad(); loss.backward(); opt.step()
        vl = val_loss()
        print(f'  epoch {ep + 1}: val loss {vl:.4f} ({time.time() - t0:.0f}s)')
        if vl < best:
            best, best_state, wait = vl, copy.deepcopy(enc.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    enc.load_state_dict(best_state)
    return enc


def extract_h(enc):
    enc.eval(); out = []
    with torch.no_grad():
        for i in range(0, N, 4096):
            xs, ts, _ = batch(np.arange(i, min(i + 4096, N)))
            out.append(enc(xs, ts).cpu().numpy())
    return np.concatenate(out)

In [10]:
# =====================================================================
# [E4-9] E4 실행
# - seed마다: 인코더 사전학습 -> h 추출 -> 거래마다 자기 (계좌, bin)의 h를 붙임
#   -> BASE_FEATS + h로 TabPFN 실행
# - h는 그 bin 이전 이력으로만 만든 값이라 현재 거래 정보가 섞이지 않습니다.
# - context 행, 평가 방식은 B0과 같습니다. 다른 것은 h를 넣었는지뿐입니다.
# - 학습한 인코더 가중치를 드라이브에 저장해 나중에 다시 확인할 수 있게 합니다.
# - 처음에는 이 셀 위에 SEEDS = [0] 을 넣어 한 번만 돌려 시간을 확인하세요.
# =====================================================================
H_COLS = [f'h{i}' for i in range(H_DIM)]
results = []

for seed in SEEDS:
    t0 = time.time()
    print(f'E4 seed {seed}: 인코더 사전학습')
    enc = train_encoder(seed)
    torch.save(enc.state_dict(), f'{MID_DIR}/enc_E4_seed{seed}.pt')

    H = pd.DataFrame(extract_h(enc), columns=H_COLS)
    H[USER], H['bin_id'] = bins[USER].values, bins['bin_id'].values

    mid = mid.drop(columns=[c for c in H_COLS if c in mid.columns])
    mid = mid.merge(H, on=[USER, 'bin_id'], how='left', validate='many_to_one')
    assert mid[H_COLS].notna().all().all()

    r = run_tabpfn(BASE_FEATS + H_COLS, seed)
    results.append({'model': 'E4', 'seed': seed, **r})
    print(f'E4 seed {seed}: val PR-AUC {r["val_pr_auc"]:.4f} | test PR-AUC {r["test_pr_auc"]:.4f} '
          f'| P {r["test_precision"]:.4f} | R {r["test_recall"]:.4f}  ({time.time() - t0:.0f}s)')

E4 seed 0: 인코더 사전학습
  epoch 1: val loss 0.1971 (13s)
  epoch 2: val loss 0.1927 (10s)
  epoch 3: val loss 0.1918 (10s)
  epoch 4: val loss 0.1908 (10s)
  epoch 5: val loss 0.1884 (10s)
  epoch 6: val loss 0.1881 (10s)
  epoch 7: val loss 0.1892 (10s)
  epoch 8: val loss 0.1910 (10s)
  epoch 9: val loss 0.1888 (10s)


tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

E4 seed 0: val PR-AUC 0.5712 | test PR-AUC 0.6080 | P 0.4578 | R 0.9272  (345s)
E4 seed 1: 인코더 사전학습
  epoch 1: val loss 0.1989 (10s)
  epoch 2: val loss 0.1925 (10s)
  epoch 3: val loss 0.1911 (10s)
  epoch 4: val loss 0.1902 (10s)
  epoch 5: val loss 0.1881 (10s)
  epoch 6: val loss 0.1877 (10s)
  epoch 7: val loss 0.1874 (10s)
  epoch 8: val loss 0.1876 (10s)
  epoch 9: val loss 0.1874 (10s)
  epoch 10: val loss 0.1891 (10s)
E4 seed 1: val PR-AUC 0.5886 | test PR-AUC 0.6454 | P 0.4299 | R 0.9905  (333s)
E4 seed 2: 인코더 사전학습
  epoch 1: val loss 0.1978 (10s)
  epoch 2: val loss 0.1916 (10s)
  epoch 3: val loss 0.1914 (10s)
  epoch 4: val loss 0.1905 (10s)
  epoch 5: val loss 0.1955 (10s)
  epoch 6: val loss 0.1882 (10s)
  epoch 7: val loss 0.1884 (10s)
  epoch 8: val loss 0.1866 (10s)
  epoch 9: val loss 0.1894 (10s)
  epoch 10: val loss 0.1880 (10s)
  epoch 11: val loss 0.1876 (10s)
E4 seed 2: val PR-AUC 0.5734 | test PR-AUC 0.5884 | P 0.4894 | R 0.8766  (343s)


In [11]:
# =====================================================================
# [E4-10] 결과 저장과 B0 비교
# - E4 결과를 results_E4.csv로 저장합니다.
# - results_B0.csv를 읽어 같은 표로 비교합니다.
#   B0을 Precision 추가 전에 돌렸다면 B0의 test_precision 칸은 비어 있습니다.
# =====================================================================
res = pd.DataFrame(results)
res.to_csv(f'{MID_DIR}/results_E4.csv', index=False)

b0 = pd.read_csv(f'{MID_DIR}/results_B0.csv')
both = pd.concat([b0, res], ignore_index=True)
cols = [c for c in ['val_pr_auc', 'test_pr_auc', 'test_f1', 'test_precision', 'test_recall'] if c in both]

print(both[['model', 'seed'] + cols].round(4).to_string(index=False))
print()
print(both.groupby('model')[cols].agg(['mean', 'std']).round(4).T)

model  seed  val_pr_auc  test_pr_auc  test_f1  test_precision  test_recall
   B0     0      0.5238       0.5735   0.6080             NaN       0.9842
   B0     1      0.5271       0.6178   0.5996             NaN       1.0000
   B0     2      0.5569       0.5977   0.6006             NaN       0.9968
   E4     0      0.5712       0.6080   0.6130          0.4578       0.9272
   E4     1      0.5886       0.6454   0.5996          0.4299       0.9905
   E4     2      0.5734       0.5884   0.6281          0.4894       0.8766

model                    B0      E4
val_pr_auc     mean  0.5359  0.5777
               std   0.0182  0.0095
test_pr_auc    mean  0.5963  0.6139
               std   0.0222  0.0290
test_f1        mean  0.6027  0.6136
               std   0.0046  0.0143
test_precision mean     NaN  0.4591
               std      NaN  0.0297
test_recall    mean  0.9937  0.9314
               std   0.0084  0.0571


In [12]:
# =====================================================================
# [E4-11] B0 Precision 보정
# - B0은 Precision 없이 저장되었으므로 F1과 Recall로 계산합니다.
#   F1 = 2PR/(P+R) 를 P에 대해 풀면 P = F1*R / (2R - F1)
# =====================================================================
m = both['model'] == 'B0'
both.loc[m, 'test_precision'] = both.loc[m, 'test_f1'] * both.loc[m, 'test_recall'] / (2 * both.loc[m, 'test_recall'] - both.loc[m, 'test_f1'])
print(both.groupby('model')[cols].agg(['mean', 'std']).round(4).T)

model                    B0      E4
val_pr_auc     mean  0.5359  0.5777
               std   0.0182  0.0095
test_pr_auc    mean  0.5963  0.6139
               std   0.0222  0.0290
test_f1        mean  0.6027  0.6136
               std   0.0046  0.0143
test_precision mean  0.4326  0.4591
               std   0.0064  0.0297
test_recall    mean  0.9937  0.9314
               std   0.0084  0.0571
